In [ ]:
import sys
sys.path.append('${TDL_ROOT_DIR}/John/MNIST_Jan14')

from Trainer import Trainer

# FORCE the class-level root used by __init__
Trainer.root = '${TDL_ROOT_DIR}/John/MNIST_Jan14'
Trainer.results_root = '${TDL_ROOT_DIR}/John/MNIST_Jan14'  # important if exists


In [ ]:
Trainer.root = '${TDL_ROOT_DIR}/John/MNIST_Jan14'
Trainer.results_root = '${TDL_ROOT_DIR}/John/MNIST_Jan14'

In [ ]:
import os
os.chdir("${TDL_ROOT_DIR}/John/MNIST_Jan14")
print(os.getcwd())


In [ ]:
from Trainer import Trainer
import os

PROJECT_ROOT = "${TDL_ROOT_DIR}/John/MNIST_Jan14"

# patch get_betti_mat to always use absolute paths
if not hasattr(Trainer, "_orig_get_betti_mat"):
    Trainer._orig_get_betti_mat = Trainer.get_betti_mat

def patched_get_betti_mat(self, dir_name, *args, **kwargs):
    # force absolute root
    self.root = PROJECT_ROOT
    return Trainer._orig_get_betti_mat(self, dir_name, *args, **kwargs)

Trainer.get_betti_mat = patched_get_betti_mat


In [ ]:
import json
from Trainer import Trainer

# Save original method
if not hasattr(Trainer, "_original_all_bootstrap_stats"):
    Trainer._original_all_bootstrap_stats = Trainer.all_bootstrap_stats

def patched_all_bootstrap_stats(*args, **kwargs):
    out = Trainer._original_all_bootstrap_stats(*args, save=False, **kwargs)

    filename = kwargs.get("filename", "all_models")
    dataset = kwargs.get("dataset", "MNIST")
    p_or_mean = "mean"
    alpha = kwargs.get("alpha", 0.05)

    new_path = (
        f"${TDL_ROOT_DIR}/John/MNIST_Jan14/"
        f"betti_data/{filename}_all_stats_{p_or_mean}_alpha{alpha}.json"
    )

    with open(new_path, "w") as f:
        json.dump(out, f, indent=2)

    print(f"✅ Saved bootstrap stats to:\n{new_path}")
    return out

Trainer.all_bootstrap_stats = patched_all_bootstrap_stats

In [ ]:
from Trainer import Trainer

# Only patch once
if not hasattr(Trainer, "_original_init"):
    Trainer._original_init = Trainer.__init__

def _patched_init(self, *args, **kwargs):
    if "root" not in kwargs or kwargs["root"] is None:
        kwargs["root"] = "${TDL_ROOT_DIR}/John/MNIST_Jan14"
    Trainer._original_init(self, *args, **kwargs)

Trainer.__init__ = _patched_init


In [ ]:
import numpy as np
from Trainer import Trainer

if not hasattr(Trainer, "_original_betti"):
    Trainer._original_betti = Trainer._betti_at_eta_one_dim

def safe_betti(diagram_dim, eta):
    arr = np.asarray(diagram_dim)
    if arr.ndim == 1 and arr.size == 2:
        arr = arr.reshape(1, 2)
    births = arr[:, 0]
    deaths = arr[:, 1]
    return np.sum((births <= eta) & (eta < deaths))

Trainer._betti_at_eta_one_dim = staticmethod(safe_betti)


In [ ]:
import os
import dill
import numpy as np

EXPECTED_LAYERS = 9  # 8 hidden + input

def empty_layer_like(layer):
    # Same number of homology dims, but empty
    return [np.empty((0, 2)) for _ in layer]

for model in os.listdir("."):
    agg = os.path.join(model, "AGG_TSS")
    if not os.path.isdir(agg):
        continue

    for fname in os.listdir(agg):
        if not fname.endswith(".dill"):
            continue

        path = os.path.join(agg, fname)
        with open(path, "rb") as f:
            diagram = dill.load(f)

        if len(diagram) >= EXPECTED_LAYERS:
            continue

        # Pad with empty layers
        while len(diagram) < EXPECTED_LAYERS:
            diagram.append(empty_layer_like(diagram[0]))

        with open(path, "wb") as f:
            dill.dump(diagram, f)

        print(f"✔ padded {model}/AGG_TSS/{fname} to {EXPECTED_LAYERS} layers")


In [ ]:
import os
print("Notebook CWD:", os.getcwd())


In [ ]:
import dill

path = f"{t.root}/256x8_relu/AGG_TSS/model_0.dill"
print("Opening:", path)

with open(path, "rb") as f:
    d = dill.load(f)

print("num layers:", len(d))
for i, layer in enumerate(d):
    print(i, len(layer), [arr.shape for arr in layer])


In [ ]:
data = Trainer.all_bootstrap_stats(
    filename="all_models",
    studies=[
        "256x8_leaky","512x8_leaky"
    ],
    dataset="MNIST",
    dir_name="AGG_TSS"
)


In [ ]:
import numpy as np

etas = np.linspace(0.05, 0.6, 12)

def beta0_at_eta(diagram, eta):
    # diagram[layer][dim], dim=0 for β0
    arr = diagram[3][0]  # layer 3, β0
    return np.sum((arr[:, 0] <= eta) & (eta < arr[:, 1]))

with open(f"{trainer.root}/256x8_leaky/AGG_TSS/model_0.dill", "rb") as f:
    d0 = dill.load(f)

with open(f"{trainer.root}/256x8_leaky/AGG_TSS/model_1.dill", "rb") as f:
    d1 = dill.load(f)

for eta in etas:
    print(
        f"eta={eta:.2f}  "
        f"model_0={beta0_at_eta(d0, eta)}  "
        f"model_1={beta0_at_eta(d1, eta)}"
    )


In [ ]:
for layer, stats in enumerate(data['256x8_leaky']['512x8_leaky']):
    print(
        f"Layer {layer}: "
        f"mean={stats['mean']:.3f}, "
        f"se={stats['se']:.3f}, "
        f"p_gt0={stats['p_gt0']:.3f}"
    )


In [ ]:
import jax
from Trainer import Trainer

trainer = Trainer(
    dataset='MNIST',
    hidden_dims=[256]*8,
    act_fn=jax.nn.leaky_relu,
    study_name='256x8_leaky',
    root='${TDL_ROOT_DIR}/John/MNIST_Jan14'
)

bm = trainer.get_betti_mat(dir_name='AGG_TSS')
print(bm.shape)
print(bm[:3])


In [ ]:
Trainer.get_tsc(
    data,
    to_calculate=[r'^256x8_leaky$', r'^512x8_leaky$'],
    metric='mean',                      # 👈 THIS IS THE KEY
    title=r'Model size comparison (Leaky)',
    legend=['256×8', '512×8'],
    legend_title='Architecture',
    plot=True,
    save=False,
    filename='256_vs_512_x8_TSS'
)
